# 1. Thu thập và đọc dữ liệu

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, udf
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_curve, auc
import warnings
warnings.filterwarnings("ignore")

spark = SparkSession.builder.appName("Stacking_Classifier").config("spark.driver.memory", "4g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.parquet("../data/processed/citation_ml_dataset.parquet")

# 2. Thống kê dữ liệu

In [2]:
print("Số lượng dòng:", df.count())
print("Số lượng cột:", len(df.columns))
df.printSchema()

# 3. Trực quan hóa dữ liệu

In [3]:
label_counts = df.groupBy("label").count().toPandas()
plt.figure(figsize=(6, 4))
sns.barplot(x="label", y="count", data=label_counts, palette=["blue", "red"])
plt.title("Phân phối nhãn")
plt.show()

# 4. Tiền xử lý dữ liệu và chia tập dữ liệu

In [4]:
features = ["citation_velocity", "pagerank_score", "paper_age"]
target = "label"

assembler = VectorAssembler(inputCols=features, outputCol="raw_features")
assembled_df = assembler.transform(df)

train_data = assembled_df.filter(assembled_df["temporal_split"] == "train")
val_data = assembled_df.filter(assembled_df["temporal_split"] == "valid")
test_data = assembled_df.filter(assembled_df["temporal_split"] == "test")

print("Train:", train_data.count())
print("Validation:", val_data.count())
print("Test:", test_data.count())

scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", withMean=False, withStd=True)
scaler_model = scaler.fit(train_data)

train_scaled = scaler_model.transform(train_data)
val_scaled = scaler_model.transform(val_data)
test_scaled = scaler_model.transform(test_data)

In [5]:
counts = {r["label"]: r["count"] for r in train_scaled.groupBy("label").count().collect()}
total = sum(counts.values())
weight_0 = total / (2.0 * counts[0.0])
weight_1 = total / (2.0 * counts[1.0])
train_weighted = train_scaled.withColumn("class_weight", when(col("label") == 0.0, weight_0).otherwise(weight_1))

# 5. Model Training (Huấn luyện Base Models: Random Forest & GBT)

In [6]:
rf = RandomForestClassifier(featuresCol="scaled_features", labelCol=target, numTrees=30, maxDepth=5, seed=42, weightCol="class_weight")
rf_model = rf.fit(train_weighted)

gbt = GBTClassifier(featuresCol="scaled_features", labelCol=target, maxDepth=4, maxIter=15, seed=42, weightCol="class_weight")
gbt_model = gbt.fit(train_weighted)

In [7]:
get_prob_1 = udf(lambda v: float(v[1]), DoubleType())

def extract_meta(df_in):
    rf_p = rf_model.transform(df_in).select("paper_id", target, get_prob_1("probability").alias("rf_prob"))
    gbt_p = gbt_model.transform(df_in).select("paper_id", get_prob_1("probability").alias("gbt_prob"))
    return rf_p.join(gbt_p, on="paper_id")

train_meta = extract_meta(train_weighted)
val_meta = extract_meta(val_scaled)
test_meta = extract_meta(test_scaled)

meta_assembler = VectorAssembler(inputCols=["rf_prob", "gbt_prob"], outputCol="meta_features")
train_meta_vec = meta_assembler.transform(train_meta)
val_meta_vec = meta_assembler.transform(val_meta)
test_meta_vec = meta_assembler.transform(test_meta)

meta_learner = LogisticRegression(featuresCol="meta_features", labelCol=target, maxIter=20)
stacking_model = meta_learner.fit(train_meta_vec)

# 6. Model Validation (Kiểm định trên tập Validation)

In [8]:
val_preds = stacking_model.transform(val_meta_vec)
eval_pr = BinaryClassificationEvaluator(labelCol=target, rawPredictionCol="rawPrediction", metricName="areaUnderPR")
eval_roc = BinaryClassificationEvaluator(labelCol=target, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
print("Validation PR-AUC:", eval_pr.evaluate(val_preds))
print("Validation ROC-AUC:", eval_roc.evaluate(val_preds))

Validation PR-AUC: 0.6613
Validation ROC-AUC: 0.9145


# 7. Model Testing (Đánh giá trên tập Test)

In [9]:
test_preds = stacking_model.transform(test_meta_vec)
print("Test PR-AUC:", eval_pr.evaluate(test_preds))
print("Test ROC-AUC:", eval_roc.evaluate(test_preds))

Test PR-AUC: 0.5412
Test ROC-AUC: 0.7888


In [10]:
test_preds.select("paper_id", "label", "prediction", "rawPrediction").show(5)

In [11]:
acc_evaluator = MulticlassClassificationEvaluator(labelCol=target, predictionCol="prediction", metricName="accuracy")
f1_evaluator = MulticlassClassificationEvaluator(labelCol=target, predictionCol="prediction", metricName="f1")
print("1. Accuracy:", acc_evaluator.evaluate(test_preds))
print("2. F1-Score:", f1_evaluator.evaluate(test_preds))

1. Accuracy: 0.8883
2. F1-Score: 0.8617


# 8. Trích xuất kết quả dự đoán

In [12]:
get_prob = udf(lambda v: float(v[1]), DoubleType())
test_preds_prob = test_preds.withColumn("prob_1", get_prob(col("probability")))
results_pd = test_preds_prob.select("label", "prediction", "prob_1").toPandas()
y_true = results_pd["label"]
y_pred = results_pd["prediction"]
y_score = results_pd["prob_1"]

# 9. Biểu đồ 1: Confusion Matrix

In [13]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[0, 1], yticklabels=[0, 1])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Stacking Classifier")
plt.show()

# 10. Biểu đồ 2: Precision-Recall Curve

In [14]:
precision, recall, _ = precision_recall_curve(y_true, y_score)
pr_auc_val = auc(recall, precision)
plt.figure(figsize=(6, 4))
plt.plot(recall, precision, color="purple", label=f"PR-AUC = {pr_auc_val:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Stacking Classifier")
plt.legend()
plt.show()

# 11. Biểu đồ 3: ROC-AUC Curve

In [15]:
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc_val = auc(fpr, tpr)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color="blue", label=f"ROC-AUC = {roc_auc_val:.4f}")
plt.plot([0, 1], [0, 1], color="red", linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Stacking Classifier")
plt.legend()
plt.show()

In [16]:
spark.stop()